# ML-08 — Capstone Modeling Lane

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

## 1. Method choice and why

**My lane:** Ranking Signal Analysis which safe content and search signals are associated with visibility.

**Target:** `is_declining_label` (visibility declining vs. not), the same label I used to sanity-check signals in my Week-4 baseline. This is an observed, computed label (`trend_direction == "down"`) never derived from the future, never a feature.

**Method:** My question is "yes/no with an observed label" — per the toolkit, that starts with **Logistic Regression**, then **Random Forest**. I'm using both:
- Logistic Regression first, because it's readable — the coefficients themselves are a plain-language answer to "which signals are associated with visibility."
- Random Forest second, to see if a stronger, nonlinear model actually earns its complexity over the simple one, per "add complexity only when the comparison earns it."

I'm also running **permutation importance** on both, because my lane is explicitly about "which signals"not just whether a model can predict the label, but what it's leaning on to do so. That's the direct toolkit fit for a "what drives X" question.

**What I will NOT use as features:** `trend_direction`, `trend_pct` (these define the label itself using them would be leakage), and `content_id` / `client_id` (pseudonymous, grouping-only, never features per the data dictionary).

In [10]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score, precision_score

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

DATA_PATH = "data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(DATA_PATH)

df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print("rows, cols:", df.shape)
print("target base rate:", round(df["is_declining_label"].mean(), 3))
print("n clients:", df["client_id"].nunique())

rows, cols: (30000, 45)
target base rate: 0.542
n clients: 32


In [11]:
!git clone https://github.com/HasnainRaza16/flyrank-ml-Hasnain.git
%cd flyrank-ml-Hasnain

Cloning into 'flyrank-ml-Hasnain'...
remote: Enumerating objects: 127, done.
remote: Counting objects: 100% (127/127), done.
remote: Compressing objects: 100% (98/98), done.
remote: Total 127 (delta 39), reused 77 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (127/127), 1.86 MiB | 5.05 MiB/s, done.
Resolving deltas: 100% (39/39), done.
/content/flyrank-ml-Hasnain/flyrank-ml-Hasnain


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

## 2. Split design

**Grouped by client, not random row-level.** `client_id` is a pseudonym for grouping only (per the data dictionary) — but more importantly, rows from the same client share a lot of structure (same site conventions, same content strategy, same baseline traffic level). A random row-level split would let the model see some pages from a client in training and other pages from that *same* client in testing that's leakage through the client, not through time. `GroupShuffleSplit` on `client_id` keeps every client entirely in either train or test, never both.

**Feature exclusions (leakage discipline):**
- `trend_direction`, `trend_pct`these directly define the label. Never features.
- `impressions_last_30d`, `impressions_prev_30d` (and the matching clicks/sessions last/prev_30d columns) `trend_pct` is a deterministic formula of `impressions_last_30d` and `impressions_prev_30d`. Including these would let the model reconstruct the label almost exactly through arithmetic, not genuine signal. Excluded.
- `content_id`, `client_id`pseudonymous IDs, grouping/joins only, never features.
- `provider_used`, `model_used` — the data dictionary explicitly marks these "Not a model feature."
- Pre-built tier columns (`age_tier`, `freshness_tier`, `position_tier`, etc.)these are just bucketed versions of raw numeric columns I'm already including (e.g. `position_tier` is a bucketed `avg_position`). Keeping both would double-count the same signal, so I use the raw numerics only.

**Missingness handling:** several keyword-context columns (`search_volume`, `competition`, `cpc`, `word_count`, `char_count`) are missing along `content_type` lines, not randomly. Per the dictionary's warning, a blind `fillna(0)` would silently encode content type into the features. I add explicit `has_*` flags before filling, so "missing" stays visible to the model as its own signal rather than being confused with a real zero.

In [12]:
# --- Safe feature set (current-window, non-label, non-ID) ---
numeric_features = [
    "impressions_90d", "clicks_90d", "pageviews_90d", "sessions_90d", "users_90d",
    "engaged_sessions_90d", "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
    "word_count", "char_count", "content_age_days", "days_since_last_update",
    "search_volume", "competition", "cpc",
]
categorical_features = ["content_type", "main_intent", "competition_level"]

# missingness flags BEFORE filling, so "no keyword data" stays visible as its own signal
df["has_keyword_data"] = df["search_volume"].notna().astype(int)
df["has_word_count"] = df["word_count"].notna().astype(int)

numeric_features += ["has_keyword_data", "has_word_count"]

# fill numeric NaNs with 0 (now safe: has_* flags preserve the "missing" information)
df[numeric_features] = df[numeric_features].fillna(0)
df[categorical_features] = df[categorical_features].fillna("unknown")

X = pd.get_dummies(df[numeric_features + categorical_features], columns=categorical_features, drop_first=True)
y = df["is_declining_label"]
groups = df["client_id"]

# --- Grouped split: every client entirely in train OR test, never both ---
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=RANDOM_SEED)
train_idx, test_idx = next(gss.split(X, y, groups=groups))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print("train rows:", len(X_train), "| test rows:", len(X_test))
print("train clients:", df.iloc[train_idx]["client_id"].nunique(), "| test clients:", df.iloc[test_idx]["client_id"].nunique())
print("overlap check (should be 0):", len(set(df.iloc[train_idx]["client_id"]) & set(df.iloc[test_idx]["client_id"])))
print("train base rate:", round(y_train.mean(), 3), "| test base rate:", round(y_test.mean(), 3))

train rows: 22885 | test rows: 7115
train clients: 24 | test clients: 8
overlap check (should be 0): 0
train base rate: 0.55 | test base rate: 0.517


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

## 3. Train + compare vs my baseline

I recompute my Week-4 baseline rule here (same formula, same thresholds) so it can be scored on the exact same held-out test rows as the model same data, same split, same metric, as the skill requires. I compare all three (baseline, Logistic Regression, Random Forest) at several values of K, plus overall ROC AUC, plus the base rate for context.

In [13]:
# --- Recreate Week-4 baseline rule, scored on the SAME test rows ---
tier_median_ctr = df.loc[df["position_tier"] != "no_data"].groupby("position_tier")["ctr"].median()
df["stale"] = (df["days_since_last_update"] >= 90).astype(int)
df["visible"] = (df["impressions_90d"] >= 500).astype(int)
df["tier_median_ctr"] = df["position_tier"].map(tier_median_ctr).fillna(np.inf)
df["ctr_gap"] = ((df["position_tier"] != "no_data") & (df["ctr"] < df["tier_median_ctr"])).astype(int)
df["baseline_score"] = df["stale"] * df["visible"] * df["ctr_gap"] * df["impressions_90d"]

baseline_score_test = df.iloc[test_idx]["baseline_score"].values

# --- Train Logistic Regression (scaled, so lbfgs converges cleanly) ---
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

logreg = LogisticRegression(max_iter=2000, random_state=RANDOM_SEED)
logreg.fit(X_train_scaled, y_train)
logreg_score_test = logreg.predict_proba(X_test_scaled)[:, 1]

# --- Train Random Forest (no scaling needed — tree-based, scale-invariant) ---
rf = RandomForestClassifier(n_estimators=300, max_depth=8, random_state=RANDOM_SEED, n_jobs=-1)
rf.fit(X_train, y_train)
rf_score_test = rf.predict_proba(X_test)[:, 1]

print("models trained. test set size:", len(y_test))


models trained. test set size: 7115


In [14]:
def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

y_test_arr = y_test.values
ks = [20, 50, 100, 200]

rows = []
for k in ks:
    rows.append({
        "K": k,
        "baseline_precision@K": round(precision_at_k(baseline_score_test, y_test_arr, k), 3),
        "logreg_precision@K": round(precision_at_k(logreg_score_test, y_test_arr, k), 3),
        "rf_precision@K": round(precision_at_k(rf_score_test, y_test_arr, k), 3),
    })

comparison_table = pd.DataFrame(rows)
comparison_table["base_rate"] = round(y_test_arr.mean(), 3)

print("=== MODEL vs BASELINE COMPARISON (same test split, same metric) ===")
print(comparison_table.to_string(index=False))
print()
print("ROC AUC — LogReg:", round(roc_auc_score(y_test, logreg_score_test), 3))
print("ROC AUC — Random Forest:", round(roc_auc_score(y_test, rf_score_test), 3))

=== MODEL vs BASELINE COMPARISON (same test split, same metric) ===
  K  baseline_precision@K  logreg_precision@K  rf_precision@K  base_rate
 20                 0.500               0.850           0.500      0.517
 50                 0.520               0.800           0.520      0.517
100                 0.530               0.740           0.500      0.517
200                 0.545               0.705           0.545      0.517

ROC AUC — LogReg: 0.589
ROC AUC — Random Forest: 0.604


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

## 4. Errors and interpretation

I run permutation importance on Logistic Regression.The model that actually won the comparison in Section 3 since my lane's question is "which signals," and the skill ties importance-checking to exactly that question shape.

In [15]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(logreg, X_test_scaled, y_test, n_repeats=10,
                               random_state=RANDOM_SEED, scoring='roc_auc')
imp_df = pd.DataFrame({
    'feature': X.columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)

print(imp_df.head(10).to_string(index=False))

              feature  importance_mean  importance_std
days_with_impressions         0.049829        0.003466
   days_with_sessions         0.034207        0.002105
            users_90d         0.023716        0.001756
     content_age_days         0.018524        0.001628
         avg_position         0.009171        0.001420
     has_keyword_data         0.004803        0.002260
          scroll_rate         0.004728        0.002656
           char_count         0.002770        0.001078
       has_word_count         0.002605        0.001146
           word_count         0.002285        0.001167


**Top 3 features by permutation importance: `days_with_impressions`, `days_with_sessions`, `users_90d`.**

These make sense, and aren't suspiciously perfect (importance scores are modest, 0.02–0.05 drop in AUC not a single feature near-perfectly explaining the label, which would smell like leakage). `days_with_impressions` and `days_with_sessions` measure how *consistently* a page shows up in search/analytics across the 90-day window — a page that was visible most days early on but goes quiet later would show both a lower count here and a declining trend, so this is a plausible, honest correlate of visibility loss, not a reconstruction of the label itself.

**Where the model is wrong-false positives (confidently predicted declining, actually wasn't):**
Several of these are pages with a real position/traffic gap between `days_with_impressions` (often 60–90, i.e. shown almost every day) and `days_with_sessions` (as low as 3–7) the model reads that gap as decline, but the label says otherwise. This looks like impression-without-click pages (high visibility, low engagement) that the model conflates with declining visibility, when the real issue is a different problem (CTR, not decline).

**Where the model is wrong-false negatives (confidently predicted NOT declining, actually was):**
The starkest case: a page with 83,603 impressions, position 3.4, and 802 users-by every absolute measure this looks like a strong, healthy page, so the model scores it very low risk. But the label says it's declining. This shows the model's top features (`days_with_impressions`, `users_90d`) capture *current scale*, not *direction of change* A large page that peaked earlier in the window and is now sliding down still looks "healthy" on raw volume, and the model misses that a big page can still be in decline.

**Error rate by content type:** accuracy sits close to the base rate for both `keyword article` (0.558, n=6,418) and `comparison article` (0.582, n=697) the model isn't meaningfully better or worse across content types, so the errors above aren't concentrated in one content category; they're a structural limitation of using volume/consistency signals to infer direction.

In [16]:
test_df = df.iloc[test_idx].copy()
test_df['pred_proba'] = logreg_score_test
test_df['true_label'] = y_test.values

top50 = test_df.sort_values('pred_proba', ascending=False).head(50)
print("Top-50 most confident 'declining' predictions — correct:", (top50['true_label']==1).sum(),
      "| wrong (false positives):", (top50['true_label']==0).sum())
print()

cols = ['content_id','content_type','main_intent','impressions_90d','avg_position',
        'days_with_impressions','days_with_sessions','users_90d','pred_proba']

print("=== 3 false positives (confident decline, actually stable/up) ===")
print(top50[top50['true_label']==0][cols].head(3).to_string(index=False))
print()

bottom200 = test_df.sort_values('pred_proba', ascending=True).head(200)
print("=== 3 false negatives (confident NOT declining, actually declining) ===")
print(bottom200[bottom200['true_label']==1][cols].head(3).to_string(index=False))
print()

test_df['pred_label'] = (test_df['pred_proba'] >= 0.5).astype(int)
test_df['correct'] = (test_df['pred_label'] == test_df['true_label']).astype(int)
print(test_df.groupby('content_type')['correct'].agg(['mean', 'count']))

Top-50 most confident 'declining' predictions — correct: 40 | wrong (false positives): 10

=== 3 false positives (confident decline, actually stable/up) ===
          content_id    content_type   main_intent  impressions_90d  avg_position  days_with_impressions  days_with_sessions  users_90d  pred_proba
content_374e795aab68 keyword article    commercial              235          31.0                     64                   3          3    0.921132
content_26d48a980581 keyword article informational             1266           4.6                     84                   5          5    0.905490
content_c94a53e3bfb8 keyword article informational             2164           8.1                     88                   7          7    0.903466

=== 3 false negatives (confident NOT declining, actually declining) ===
          content_id    content_type   main_intent  impressions_90d  avg_position  days_with_impressions  days_with_sessions  users_90d  pred_proba
content_8818fd6d967f keyword a

## Self-check

Before you submit, confirm each line honestly:

- [✓] Every section above is filled — markdown thinking AND the code that backs it
- [✓] The notebook runs top to bottom with no errors (Runtime → Run all)
- [✓] No client names, URLs, or private queries anywhere
- [✓] My claims use careful words: observed, measured, directional, decision-support
- [✓] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.